For your lab, you are tasked to grab a 2 csv files (with normal distribution) from the valid sources in the internet, perform a complete and detailed hypothesis test, determine what test to be used for each of your dataset. We will continue and present it in the next

*Note: Don't be discouraged if it gets hard to find a normally distributed data, it shows you that in the real world, data is not always perfect.*

1. Normality tests
2. Hypothesis testing

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (shapiro, normaltest, anderson, kstest,
                         ttest_1samp, ttest_ind, mannwhitneyu, wilcoxon)

url_ratings = 'https://raw.githubusercontent.com/prasertcbs/basic-dataset/master/Video_Games_Sales_as_at_22_Dec_2016.csv'
url_sales   = 'https://raw.githubusercontent.com/raghav-19/Video-Games-Sales-Data-Analysis/master/vgsales.csv'

df_ratings = pd.read_csv(url_ratings)
df_sales   = pd.read_csv(url_sales)

Dataset 1: Critic Scores, Metacritic

Hypothesis test: Is the average critic score different from 70?

On Metacritic a score of 70 is treated as a "good score".

alpha = 0.05

In [3]:
critic = pd.to_numeric(df_ratings['Critic_Score'], errors='coerce').dropna()
print(f'Number of Critic_Score observations: {len(critic)}')
print(critic.describe().round(2))
print(f'Skewness: {critic.skew():.3f}')
print(f'Kurtosis: {critic.kurtosis():.3f}')

Number of Critic_Score observations: 8137
count    8137.00
mean       68.97
std        13.94
min        13.00
25%        60.00
50%        71.00
75%        79.00
max        98.00
Name: Critic_Score, dtype: float64
Skewness: -0.614
Kurtosis: 0.143


In [4]:
# One-sample t-test
t_stat, t_p = ttest_1samp(critic, popmean=70)
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {t_p:.4e}')
print(f'Mean        = {critic.mean():.3f}')
print(f'Median      = {critic.median():.1f}')
ci = stats.t.interval(0.95, len(critic)-1, loc=critic.mean(), scale=stats.sem(critic))
print(f'95% CI      = ({ci[0]:.3f}, {ci[1]:.3f})')

t-statistic = -6.6810
p-value     = 2.5293e-11
Mean        = 68.968
Median      = 71.0
95% CI      = (68.665, 69.271)


In [5]:
# Wilcoxon signed rank (tests median = 70)
w_stat, w_p = wilcoxon(critic - 70)
print(f'W-statistic = {w_stat:.1f}')
print(f'p-value     = {w_p:.4e}')

W-statistic = 15178843.5
p-value     = 8.2471e-02


The one-sample t-test rejects hypothesis.

The Wilcoxon signed-rank test does not reject hypothesis.

Dataset 2: Global Sales

In [8]:
sales = pd.to_numeric(df_sales['Global_Sales'], errors='coerce').dropna()
sales = sales[sales > 0]
print(f'Global_Sales: {len(sales)}')
print(sales.describe().round(3))
print(f'Skewness: {sales.skew():.3f}')

Global_Sales: 16539
count    16539.000
mean         0.538
std          1.557
min          0.010
25%          0.060
50%          0.170
75%          0.480
max         82.740
Name: Global_Sales, dtype: float64
Skewness: 17.379


In [13]:
action = df_sales[df_sales['Genre'] == 'Action']['Global_Sales'].dropna()
sports = df_sales[df_sales['Genre'] == 'Sports']['Global_Sales'].dropna()

log_action = np.log(action)
log_sports = np.log(sports)

t_stat, t_p = ttest_ind(log_action, log_sports, equal_var=False)
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {t_p:.4e}')

u_stat, u_p = mannwhitneyu(action, sports, alternative='two-sided')
print(f'U-statistic = {u_stat:.1f}')
print(f'p-value     = {u_p:.4e}')

t-statistic = -4.1484
p-value     = 3.4011e-05
U-statistic = 3626464.0
p-value     = 3.5014e-05


Both tests reject the null at alpha = 0.05. There is evidence that the sales distributions of Action and Sports games differ.
Sports games show a higher mean while the medians are closer.